# LeetCode 1409 — Queries on a Permutation With Key bằng Python

Link đề: https://leetcode.com/problems/queries-on-a-permutation-with-key/ · Độ khó: **Medium** · Chủ đề: Array, Binary Indexed Tree, Simulation, Sqrt Decomposition

Skeleton LeetCode:

```python
class Solution:
    def processQueries(self, queries: list[int], m: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng đúng thao tác **"tìm vị trí rồi đưa lên đầu"** (move-to-front) trên `list` Python và hiểu chi phí của `index`, `pop`, `insert(0, …)`.
2. Đổi góc nhìn: thay vì giữ dãy, giữ **mảng vị trí** `pos[v]` và cập nhật dịch chuyển.
3. Dùng **Fenwick tree với vị trí ảo**: dành sẵn chỗ trống phía trước để "đưa lên đầu" chỉ là đặt vào ô trống kế tiếp — `O(log)` mỗi truy vấn.
4. Hiểu insight **"vị trí = số giá trị khác nhau được truy vấn kể từ lần cuối"** (khoảng cách ngăn xếp kiểu LRU).
5. Tránh lỗi: trả vị trí sau khi di chuyển, đánh chỉ số từ 1, quên xoá phần tử cũ, dịch sai biên, Fenwick thiếu chỗ.

## 2. Đề bài & ví dụ

Ban đầu có hoán vị `P = [1, 2, 3, …, m]`. Lần lượt xử lý `queries[i]` (từ `i = 0`):

- tìm **vị trí** (đánh số từ 0) của `queries[i]` trong `P` — đó là **kết quả** của truy vấn này;
- **đưa** phần tử đó lên **đầu** `P`.

Trả về mảng kết quả.

**Ví dụ 1:** `queries = [3,1,2,1], m = 5` → `[2,1,2,1]`.
- `P=[1,2,3,4,5]`, 3 ở vị trí 2 → `P=[3,1,2,4,5]`
- 1 ở vị trí 1 → `P=[1,3,2,4,5]`
- 2 ở vị trí 2 → `P=[2,1,3,4,5]`
- 1 ở vị trí 1 → `P=[1,2,3,4,5]`

**Ví dụ 2:** `queries = [4,1,2,2], m = 4` → `[3,1,2,0]`.

**Ví dụ 3:** `queries = [7,5,5,8,3], m = 8` → `[6,5,0,7,5]`.

**Ràng buộc:** `1 <= m <= 10^3`, `1 <= queries.length <= m`, `1 <= queries[i] <= m`.

In [1]:
import random
import time
from typing import List

examples = [([3, 1, 2, 1], 5, [2, 1, 2, 1]),
            ([4, 1, 2, 2], 4, [3, 1, 2, 0]),
            ([7, 5, 5, 8, 3], 8, [6, 5, 0, 7, 5])]
for q, m, want in examples:
    print(f"queries={q}, m={m} -> {want}")

queries=[3, 1, 2, 1], m=5 -> [2, 1, 2, 1]
queries=[4, 1, 2, 2], m=4 -> [3, 1, 2, 0]
queries=[7, 5, 5, 8, 3], m=8 -> [6, 5, 0, 7, 5]


## 3. Kiến thức nền

### 3.1 `list.index`, `pop`, `insert`

- `P.index(x)`: vị trí đầu tiên của `x`, `O(n)` (quét tuyến tính).
- `P.pop(i)`: xoá và trả về phần tử tại `i`, `O(n − i)` (dịch các phần tử phía sau).
- `P.insert(0, x)`: chèn vào đầu, `O(n)` (dịch toàn bộ).

In [2]:
P = [1, 2, 3, 4, 5]
i = P.index(3)
x = P.pop(i)
P.insert(0, x)
print("vị trí:", i, "| P sau:", P)

vị trí: 2 | P sau: [3, 1, 2, 4, 5]


### 3.2 `list(range(1, m + 1))`

Tạo hoán vị ban đầu `[1, …, m]`. Lưu ý `range(m)` cho `0 … m−1` — sai giá trị.

In [3]:
m = 5
print(list(range(1, m + 1)), "vs", list(range(m)))

[1, 2, 3, 4, 5] vs [0, 1, 2, 3, 4]


### 3.3 Mảng vị trí (nghịch đảo của hoán vị)

Nếu `P[i] = v` thì `pos[v] = i`. Biết `pos` thì tra vị trí của `v` trong `O(1)`. Dùng mảng kích thước `m + 1` (bỏ ô 0) để chỉ số trùng giá trị.

In [4]:
P = [3, 1, 2, 4, 5]
pos = [0] * (len(P) + 1)
for i, v in enumerate(P):
    pos[v] = i
print("pos[1..5] =", pos[1:], "| vị trí của 2:", pos[2])

pos[1..5] = [1, 2, 0, 3, 4] | vị trí của 2: 2


### 3.4 Fenwick tree (Binary Indexed Tree)

Hỗ trợ `update(i, delta)` và `query(i) = tổng 1..i` trong `O(log N)`; **chỉ số từ 1**. Nếu đặt `1` tại các ô đang có phần tử, thì `query(p) − 1` = số phần tử **đứng trước** ô `p` = vị trí (từ 0) của phần tử tại `p`.

In [5]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)

    def update(self, i, d):
        while i <= self.n:
            self.t[i] += d
            i += i & -i

    def query(self, i):
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

fw = Fenwick(8)
for i in [3, 5, 6]:
    fw.update(i, 1)
print("số ô có phần tử trong 1..5:", fw.query(5), "| vị trí (từ 0) của phần tử ở ô 6:", fw.query(6) - 1)

số ô có phần tử trong 1..5: 2 | vị trí (từ 0) của phần tử ở ô 6: 2


### 3.5 `set` để đếm số giá trị khác nhau

`len(set(seq))` = số giá trị phân biệt trong `seq`.

In [6]:
print(len(set([5, 5, 8, 5, 3])))

3


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3** `queries = [7,5,5,8,3]`, `m = 8`:

| i | q | P trước | vị trí | P sau |
|---|---|---|---|---|
| 0 | 7 | 1 2 3 4 5 6 **7** 8 | 6 | 7 1 2 3 4 5 6 8 |
| 1 | 5 | 7 1 2 3 4 **5** 6 8 | 5 | 5 7 1 2 3 4 6 8 |
| 2 | 5 | **5** 7 1 2 3 4 6 8 | 0 | 5 7 1 2 3 4 6 8 |
| 3 | 8 | 5 7 1 2 3 4 6 **8** | 7 | 8 5 7 1 2 3 4 6 |
| 4 | 3 | 8 5 7 1 2 **3** 4 6 | 5 | 3 8 5 7 1 2 4 6 |

Quan sát: khi đưa `q` lên đầu, **chỉ các phần tử đứng trước `q`** bị đẩy lùi một chỗ; các phần tử sau `q` giữ nguyên vị trí. Đây là cơ sở cho Cách 2 (mảng vị trí).

In [7]:
P = list(range(1, 9))
for q in [7, 5, 5, 8, 3]:
    i = P.index(q)
    before = " ".join(map(str, P))
    P.insert(0, P.pop(i))
    print(f"q={q}: [{before}] vị trí {i} -> [{' '.join(map(str, P))}]")

q=7: [1 2 3 4 5 6 7 8] vị trí 6 -> [7 1 2 3 4 5 6 8]
q=5: [7 1 2 3 4 5 6 8] vị trí 5 -> [5 7 1 2 3 4 6 8]
q=5: [5 7 1 2 3 4 6 8] vị trí 0 -> [5 7 1 2 3 4 6 8]
q=8: [5 7 1 2 3 4 6 8] vị trí 7 -> [8 5 7 1 2 3 4 6]
q=3: [8 5 7 1 2 3 4 6] vị trí 5 -> [3 8 5 7 1 2 4 6]


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng `list`

### Ý tưởng
Làm đúng như đề: `index` tìm vị trí, `pop` lấy ra, `insert(0, …)` đưa lên đầu.

In [8]:
def process_list(queries, m):
    P = list(range(1, m + 1))
    res = []
    for q in queries:
        i = P.index(q)
        res.append(i)
        P.insert(0, P.pop(i))
    return res

for q, m, want in examples:
    print(process_list(q, m), "| mong đợi", want)

[2, 1, 2, 1] | mong đợi [2, 1, 2, 1]
[3, 1, 2, 0] | mong đợi [3, 1, 2, 0]
[6, 5, 0, 7, 5] | mong đợi [6, 5, 0, 7, 5]


### Độ phức tạp và nút thắt

Gọi `n = len(queries)`. Mỗi truy vấn `O(m)` (`index` + `pop` + `insert`) ⇒ **Time `O(n·m)`** ≤ 10⁶ với ràng buộc — **chạy tốt** (các thao tác list chạy trong C rất nhanh). Space `O(m)`.

Nút thắt chỉ lộ ra khi `m`, `n` lên `10⁵` (≈ 10¹⁰ thao tác). Ta vẫn học các cách tốt hơn vì chúng là kỹ thuật quan trọng (Fenwick vị trí ảo dùng lại ở nhiều bài).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ phần đứng trước bị dịch:** khi `q` ở vị trí `p` được đưa lên đầu, mọi phần tử có vị trí `< p` tăng vị trí thêm 1, `q` thành vị trí 0, phần còn lại không đổi ⇒ cập nhật **mảng vị trí** thay vì dãy (Cách 2).

**Insight 2 — "đưa lên đầu" = "đặt vào ô trống phía trước":** dành sẵn `n` ô trống **trước** dãy ban đầu. Truy vấn thứ `i` đặt phần tử vào ô `n − i` (ô trống sát đầu nhất còn lại). Vị trí của một phần tử = **số ô có phần tử đứng trước nó** ⇒ đếm bằng Fenwick (Cách 3). Không có phép dịch nào cả!

**Insight 3 — khoảng cách kiểu LRU:** phần tử đứng trước `q` chính là những giá trị **được truy vấn sau lần cuối `q` được truy vấn** (mỗi giá trị đếm một lần). Nếu `q` chưa từng được truy vấn, phía trước nó là mọi giá trị đã truy vấn cộng với các giá trị nhỏ hơn `q` chưa từng bị chạm (Cách 4).

**Quy tắc "nếu… thì…" (Cách 2):** nếu `pos[v] < pos[q]` thì `pos[v] += 1`; sau đó `pos[q] = 0`.

## 7. Các cách tiếp theo

### Cách 2 — Mảng vị trí

**Từng đoạn:**
1. `pos[v] = v − 1` cho `v = 1..m`.
2. Với mỗi `q`: `p = pos[q]`, ghi `p` vào kết quả.
3. Với mọi `v` có `pos[v] < p`: `pos[v] += 1`.
4. `pos[q] = 0`.

In [9]:
def process_pos(queries, m):
    pos = list(range(-1, m))          # pos[v] = v - 1, pos[0] bỏ trống
    res = []
    for q in queries:
        p = pos[q]
        res.append(p)
        for v in range(1, m + 1):
            if pos[v] < p:
                pos[v] += 1
        pos[q] = 0
    return res

for q, m, want in examples:
    print(process_pos(q, m), "| mong đợi", want)

[2, 1, 2, 1] | mong đợi [2, 1, 2, 1]
[3, 1, 2, 0] | mong đợi [3, 1, 2, 0]
[6, 5, 0, 7, 5] | mong đợi [6, 5, 0, 7, 5]


Time `O(n·m)` — cùng bậc với Cách 1, nhưng không cần `index`/`insert`; ý tưởng "cập nhật vị trí thay vì dãy" là bước đệm cho Fenwick. Space `O(m)`.

### Cách 3 — Fenwick tree với vị trí ảo, `O((n + m) log(n + m))`

**Bố trí ô:** tổng `n + m` ô (đánh số từ 1).
- Ô `1 … n`: để trống, dành cho các lần "đưa lên đầu".
- Ô `n + 1 … n + m`: giá trị `1 … m` theo thứ tự ban đầu ⇒ `where[v] = n + v`.

**Từng đoạn:**
1. Fenwick đánh dấu `1` tại các ô `n + 1 … n + m`.
2. Truy vấn thứ `i` (từ 0) với giá trị `q`: `p = where[q]`; kết quả `= query(p) − 1` (số phần tử đứng trước).
3. Xoá ở ô cũ: `update(p, −1)`. Ô mới `= n − i` (ô trống sát đầu nhất): `update(n − i, +1)`, `where[q] = n − i`.

Vì `i` tăng dần nên ô mới lùi dần về phía trước: ô `n`, rồi `n − 1`, …, luôn đứng trước mọi phần tử đang có.

In [10]:
def process_fenwick(queries, m):
    n = len(queries)
    fw = Fenwick(n + m)
    where = [0] * (m + 1)
    for v in range(1, m + 1):
        where[v] = n + v
        fw.update(n + v, 1)
    res = []
    for i, q in enumerate(queries):
        p = where[q]
        res.append(fw.query(p) - 1)
        fw.update(p, -1)
        where[q] = n - i
        fw.update(n - i, 1)
    return res

for q, m, want in examples:
    print(process_fenwick(q, m), "| mong đợi", want)

[2, 1, 2, 1] | mong đợi [2, 1, 2, 1]
[3, 1, 2, 0] | mong đợi [3, 1, 2, 0]
[6, 5, 0, 7, 5] | mong đợi [6, 5, 0, 7, 5]


In [11]:
class SolutionFenwick:
    def processQueries(self, queries: List[int], m: int) -> List[int]:
        n = len(queries)
        size = n + m
        tree = [0] * (size + 1)

        def update(i, d):
            while i <= size:
                tree[i] += d
                i += i & -i

        def query(i):
            s = 0
            while i > 0:
                s += tree[i]
                i -= i & -i
            return s

        where = [0] * (m + 1)
        for v in range(1, m + 1):
            where[v] = n + v
            update(n + v, 1)
        res = []
        for i, q in enumerate(queries):
            p = where[q]
            res.append(query(p) - 1)
            update(p, -1)
            where[q] = n - i
            update(n - i, 1)
        return res

print([SolutionFenwick().processQueries(q, m) for q, m, _ in examples])

[[2, 1, 2, 1], [3, 1, 2, 0], [6, 5, 0, 7, 5]]


Time `O((n + m) log(n + m))` (khởi tạo `m` lần update + `n` truy vấn × 3 thao tác), Space `O(n + m)`.

### Cách 4 — Đếm giá trị khác nhau (insight LRU)

Với truy vấn thứ `i`, giá trị `q`:
- Nếu `q` đã được truy vấn lần cuối tại `j`: vị trí `= len(set(queries[j+1 : i]))`.
- Nếu chưa: gọi `S = set(queries[:i])`; vị trí `= (q − 1) + |{v ∈ S : v > q}|` (các giá trị nhỏ hơn `q` vẫn ở trước nó — dù có bị kéo lên đầu hay không — cộng thêm các giá trị lớn hơn `q` đã bị kéo lên trước nó).

Time `O(n²)` (mỗi truy vấn dựng một `set`), không cần giữ `P`. Giá trị chính là **cách nhìn**: move-to-front ⇔ "ngăn xếp LRU".

In [12]:
def process_distinct(queries, m):
    last = {}
    res = []
    for i, q in enumerate(queries):
        if q in last:
            res.append(len(set(queries[last[q] + 1:i])))
        else:
            S = set(queries[:i])
            res.append((q - 1) + sum(1 for v in S if v > q))
        last[q] = i
    return res

for q, m, want in examples:
    print(process_distinct(q, m), "| mong đợi", want)

[2, 1, 2, 1] | mong đợi [2, 1, 2, 1]
[3, 1, 2, 0] | mong đợi [3, 1, 2, 0]
[6, 5, 0, 7, 5] | mong đợi [6, 5, 0, 7, 5]


Kiểm tra bằng tay truy vấn cuối của Ví dụ 3 (`q = 3`, chưa từng truy vấn): `S = {7, 5, 8}`, các giá trị lớn hơn 3 trong `S` là cả 3 ⇒ vị trí `= 2 + 3 = 5` ✔.

## 8. TRACE từng bước

Trace Cách 3 (Fenwick vị trí ảo) trên **Ví dụ 1** `queries = [3,1,2,1]`, `m = 5` ⇒ `n = 4`, tổng 9 ô. Ban đầu ô 5..9 chứa 1..5.

| i | q | ô cũ `p` | `query(p)` | kết quả | ô mới `n − i` | dãy ô 1..9 sau (`.` = trống) |
|---|---|---|---|---|---|---|
| — | | | | | | `. . . . 1 2 3 4 5` |
| 0 | 3 | 7 | 3 | 2 | 4 | `. . . 3 1 2 . 4 5` |
| 1 | 1 | 5 | 2 | 1 | 3 | `. . 1 3 . 2 . 4 5` |
| 2 | 2 | 6 | 3 | 2 | 2 | `. 2 1 3 . . . 4 5` |
| 3 | 1 | 3 | 2 | 1 | 1 | `1 2 . 3 . . . 4 5` |

In [13]:
def trace_fenwick(queries, m):
    n = len(queries)
    fw = Fenwick(n + m)
    where = [0] * (m + 1)
    slot = ["."] * (n + m + 1)
    for v in range(1, m + 1):
        where[v] = n + v; fw.update(n + v, 1); slot[n + v] = str(v)
    print("ban đầu:", " ".join(slot[1:]))
    res = []
    for i, q in enumerate(queries):
        p = where[q]
        before = fw.query(p)
        res.append(before - 1)
        fw.update(p, -1); slot[p] = "."
        new = n - i
        where[q] = new; fw.update(new, 1); slot[new] = str(q)
        print(f"i={i} q={q}: ô cũ {p}, query({p})={before} -> kết quả {before - 1}; ô mới {new} | {' '.join(slot[1:])}")
    return res

print("kết quả:", trace_fenwick([3, 1, 2, 1], 5))

ban đầu: . . . . 1 2 3 4 5
i=0 q=3: ô cũ 7, query(7)=3 -> kết quả 2; ô mới 4 | . . . 3 1 2 . 4 5
i=1 q=1: ô cũ 5, query(5)=2 -> kết quả 1; ô mới 3 | . . 1 3 . 2 . 4 5
i=2 q=2: ô cũ 6, query(6)=3 -> kết quả 2; ô mới 2 | . 2 1 3 . . . 4 5
i=3 q=1: ô cũ 3, query(3)=2 -> kết quả 1; ô mới 1 | 1 2 . 3 . . . 4 5
kết quả: [2, 1, 2, 1]


Đọc dãy ô bỏ dấu `.` từ trái sang phải chính là `P` hiện tại: sau bước cuối là `1 2 3 4 5` — khớp với mô phỏng ở mục 2.

Trace Cách 2 (mảng vị trí) trên Ví dụ 2:

In [14]:
def trace_pos(queries, m):
    pos = list(range(-1, m))
    for q in queries:
        p = pos[q]
        shifted = [v for v in range(1, m + 1) if pos[v] < p]
        for v in shifted:
            pos[v] += 1
        pos[q] = 0
        P = sorted(range(1, m + 1), key=lambda v: pos[v])
        print(f"q={q}: vị trí {p}, dịch lùi {shifted} -> pos[1..{m}]={pos[1:]} (P={P})")

trace_pos([4, 1, 2, 2], 4)

q=4: vị trí 3, dịch lùi [1, 2, 3] -> pos[1..4]=[1, 2, 3, 0] (P=[4, 1, 2, 3])
q=1: vị trí 1, dịch lùi [4] -> pos[1..4]=[0, 2, 3, 1] (P=[1, 4, 2, 3])
q=2: vị trí 2, dịch lùi [1, 4] -> pos[1..4]=[1, 0, 3, 2] (P=[2, 1, 4, 3])
q=2: vị trí 0, dịch lùi [] -> pos[1..4]=[1, 0, 3, 2] (P=[2, 1, 4, 3])


## 9. Code nộp LeetCode

Với `m ≤ 1000`, mô phỏng list là ngắn và đủ nhanh; dưới đây là bản **Fenwick** (tổng quát, `O((n + m) log)`), kèm bản list ngắn gọn ở Cách 1.

In [15]:
from typing import List

class Solution:
    def processQueries(self, queries: List[int], m: int) -> List[int]:
        n = len(queries)
        size = n + m                      # n ô trống phía trước + m ô cho hoán vị ban đầu
        tree = [0] * (size + 1)

        def update(i: int, d: int) -> None:
            while i <= size:
                tree[i] += d
                i += i & -i

        def query(i: int) -> int:         # số ô có phần tử trong 1..i
            s = 0
            while i > 0:
                s += tree[i]
                i -= i & -i
            return s

        where = [0] * (m + 1)
        for v in range(1, m + 1):
            where[v] = n + v
            update(n + v, 1)
        res = []
        for i, q in enumerate(queries):
            p = where[q]
            res.append(query(p) - 1)      # số phần tử đứng trước q
            update(p, -1)                 # nhấc q ra
            where[q] = n - i              # đặt vào ô trống sát đầu nhất
            update(n - i, 1)
        return res

for q, m, want in examples:
    assert Solution().processQueries(q, m) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Cách 1** làm đúng định nghĩa.

**Cách 2.** Bất biến: `pos[v]` = vị trí của `v` trong `P` hiện tại. Khi đưa `q` (ở vị trí `p`) lên đầu, dãy mới là `[q] + P[0:p] + P[p+1:]`: phần tử ở vị trí `< p` lùi 1, phần tử ở vị trí `> p` giữ nguyên, `q` về 0 — đúng như cập nhật.

**Cách 3.** Bất biến: (a) mỗi giá trị chiếm đúng một ô, ô có phần tử được đánh dấu 1 trong Fenwick; (b) đọc các ô có phần tử từ trái sang phải cho đúng `P`. Ban đầu đúng. Ở truy vấn `i`, các ô `1 … n − i` còn trống (đã dùng các ô `n, n − 1, …, n − i + 1`), nên ô `n − i` đứng trước **mọi** ô đang có phần tử ⇒ đặt `q` vào đó đúng là "đưa lên đầu", thứ tự các phần tử khác giữ nguyên ⇒ (b) được bảo toàn. Vị trí của `q` = số ô có phần tử đứng trước ô `p` = `query(p) − 1`. Có đủ ô vì có đúng `n` truy vấn.

**Cách 4.** Mỗi lần truy vấn, giá trị đó lên đầu. Sau lần truy vấn cuối của `q` tại `j`, các giá trị được truy vấn trong `(j, i)` lần lượt nhảy lên trước `q` (mỗi giá trị chỉ nằm trước `q` một lần dù bị truy vấn nhiều lần), còn giá trị không được truy vấn thì vẫn ở sau `q` ⇒ vị trí = số giá trị phân biệt trong khoảng đó. Nếu `q` chưa từng được truy vấn, `q` còn ở "phần gốc" theo thứ tự tăng; phía trước nó: mọi giá trị đã truy vấn (đều ở phần đầu) và các giá trị `< q` chưa truy vấn ⇒ `|S| + (q − 1 − |{v∈S: v<q}|) = (q − 1) + |{v∈S: v>q}|`.

## 11. Phân tích Time / Space

Gọi `n = len(queries) ≤ m ≤ 1000`.

| Cách | Time | Space | Với n = m = 1000 |
|---|---|---|---|
| 1. Mô phỏng list | `O(n·m)` (thao tác trong C) | `O(m)` | rất nhanh |
| 2. Mảng vị trí | `O(n·m)` (vòng Python) | `O(m)` | ~10⁶ phép so sánh |
| 3. Fenwick vị trí ảo | `O((n + m) log(n + m))` | `O(n + m)` | ~6·10⁴ thao tác |
| 4. Đếm giá trị khác nhau | `O(n²)` | `O(n)` | ~10⁶ |

Khi `n, m ~ 10⁵` chỉ còn Cách 3 khả thi (xem 12.3).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {
    "list": process_list,
    "pos": process_pos,
    "fenwick": process_fenwick,
    "fenwick_submit": lambda q, m: SolutionFenwick().processQueries(q, m),
    "distinct": process_distinct,
    "submit": lambda q, m: Solution().processQueries(q, m),
}
edge = [
    ([1], 1, [0]),                      # m nhỏ nhất
    ([1, 1, 1], 3, [0, 0, 0]),          # luôn ở đầu
    ([3, 3], 3, [2, 0]),
    ([3, 2, 1], 3, [2, 2, 2]),          # mỗi lần lấy phần tử cuối
    ([1, 2, 3], 3, [0, 1, 2]),
    ([2, 3, 2], 3, [1, 2, 1]),
    ([5, 4, 3, 2, 1], 5, [4, 4, 4, 4, 4]),
]
for q, m, want in edge:
    assert process_list(q, m) == want, (q, m, process_list(q, m))
for name, f in ALL.items():
    for q, m, want in examples + edge:
        assert f(q, m) == want, (name, q, m)
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng mô phỏng)")

Mọi cách đúng với 10 test (nhãn edge kiểm chứng bằng mô phỏng)


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, đúng ràng buộc `n ≤ m`)

In [17]:
rng = random.Random(1409)
for _ in range(1500):
    m = rng.randint(1, 30)
    n = rng.randint(1, m)
    q = [rng.randint(1, m) for _ in range(n)]
    want = process_list(q, m)
    for name, f in ALL.items():
        assert f(q, m) == want, (name, q, m)
print("1500 bộ ngẫu nhiên: mọi cách khớp mô phỏng list")

1500 bộ ngẫu nhiên: mọi cách khớp mô phỏng list


### 12.3 Hiệu năng: ràng buộc `m = 1000` và thử `m = n = 2·10^4` (ngoài ràng buộc)

In [18]:
m = 1000
q = [rng.randint(1, m) for _ in range(m)]
for name in ALL:
    t = time.perf_counter(); r = ALL[name](q, m)
    print(f"m=1000  {name:15s}: {time.perf_counter() - t:.3f}s")
    assert r == process_list(q, m)

m = 2 * 10**4
q = [rng.randint(1, m) for _ in range(m)]
for name in ["list", "fenwick", "submit"]:
    t = time.perf_counter(); r = ALL[name](q, m)
    print(f"m=2e4   {name:15s}: {time.perf_counter() - t:.3f}s")
assert process_fenwick(q, m) == Solution().processQueries(q, m)

m=1000  list           : 0.004s


m=1000  pos            : 0.045s


m=1000  fenwick        : 0.002s
m=1000  fenwick_submit : 0.002s


m=1000  distinct       : 0.015s


m=1000  submit         : 0.002s


m=2e4   list           : 1.475s
m=2e4   fenwick        : 0.059s
m=2e4   submit         : 0.059s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ghi vị trí **sau** khi đã đưa lên đầu

In [19]:
def wrong_after_move(queries, m):
    P = list(range(1, m + 1))
    res = []
    for q in queries:
        P.insert(0, P.pop(P.index(q)))
        res.append(P.index(q))                 # SAI: luôn là 0
    return res

for q, m, want in examples:
    print(q, "| sai:", wrong_after_move(q, m), "| đúng:", want)

[3, 1, 2, 1] | sai: [0, 0, 0, 0] | đúng: [2, 1, 2, 1]
[4, 1, 2, 2] | sai: [0, 0, 0, 0] | đúng: [3, 1, 2, 0]
[7, 5, 5, 8, 3] | sai: [0, 0, 0, 0, 0] | đúng: [6, 5, 0, 7, 5]


**Sửa:** ghi vị trí **trước** khi di chuyển.

### Lỗi 2 — Đánh số vị trí từ 1

In [20]:
def wrong_one_based(queries, m):
    P = list(range(1, m + 1))
    res = []
    for q in queries:
        i = P.index(q)
        res.append(i + 1)                      # SAI: đề đánh số từ 0
        P.insert(0, P.pop(i))
    return res

for q, m, want in examples:
    print(q, "| sai:", wrong_one_based(q, m), "| đúng:", want)

[3, 1, 2, 1] | sai: [3, 2, 3, 2] | đúng: [2, 1, 2, 1]
[4, 1, 2, 2] | sai: [4, 2, 3, 1] | đúng: [3, 1, 2, 0]
[7, 5, 5, 8, 3] | sai: [7, 6, 1, 8, 6] | đúng: [6, 5, 0, 7, 5]


**Sửa:** `list.index` đã trả chỉ số từ 0 — dùng thẳng.

### Lỗi 3 — Chèn lên đầu nhưng quên xoá phần tử cũ

`P.insert(0, q)` mà không `pop` ⇒ `P` có bản sao và dài dần; `index` vẫn tìm bản **đầu tiên** nên lỗi chỉ lộ ra ở một số truy vấn.

In [21]:
def wrong_no_remove(queries, m):
    P = list(range(1, m + 1))
    res = []
    for q in queries:
        res.append(P.index(q))
        P.insert(0, q)                         # SAI: không xoá bản cũ
    return res, len(P)

for q, m, want in examples:
    got, size = wrong_no_remove(q, m)
    print(q, "| sai:", got, f"(len P = {size})", "| đúng:", want)

[3, 1, 2, 1] | sai: [2, 1, 3, 1] (len P = 9) | đúng: [2, 1, 2, 1]
[4, 1, 2, 2] | sai: [3, 1, 3, 0] (len P = 8) | đúng: [3, 1, 2, 0]
[7, 5, 5, 8, 3] | sai: [6, 5, 0, 10, 6] (len P = 13) | đúng: [6, 5, 0, 7, 5]


Ví dụ 1: truy vấn thứ hai `1` — `P = [3,1,2,3,4,5]` nên vẫn ra 1 (tình cờ đúng), nhưng truy vấn thứ ba `2` ở vị trí 3 thay vì 2 vì bản sao của `3` còn nằm phía sau. **Sửa:** `P.insert(0, P.pop(i))`.

### Lỗi 4 — Mảng vị trí: gán `pos[q] = 0` **trước** khi dịch

Thứ tự hai thao tác quan trọng: nếu gán `pos[q] = 0` trước, thì ở vòng dịch `q` (giờ có vị trí `0 < p`) cũng bị đẩy lùi thành 1 ⇒ hai phần tử cùng "vị trí 1", không phần tử nào ở vị trí 0.

In [22]:
def wrong_shift_order(queries, m):
    pos = list(range(-1, m))
    res = []
    for q in queries:
        p = pos[q]
        res.append(p)
        pos[q] = 0                             # SAI: gán trước khi dịch
        for v in range(1, m + 1):
            if pos[v] < p:                     # q (pos = 0) cũng bị dịch thành 1
                pos[v] += 1
    return res

for q, m, want in examples:
    print(q, "| sai:", wrong_shift_order(q, m), "| đúng:", want)

[3, 1, 2, 1] | sai: [2, 1, 2, 2] | đúng: [2, 1, 2, 1]
[4, 1, 2, 2] | sai: [3, 1, 2, 1] | đúng: [3, 1, 2, 0]
[7, 5, 5, 8, 3] | sai: [6, 5, 1, 7, 5] | đúng: [6, 5, 0, 7, 5]


**Sửa:** dịch các phần tử có `pos[v] < p` **trước**, rồi mới `pos[q] = 0`.

### Lỗi 5 — Fenwick không chừa đủ ô phía trước

Bố trí ô phải khớp nhau: nếu chỉ chừa `n − 1` ô trống (đặt `v` ở ô `n − 1 + v`) nhưng vẫn dùng công thức ô mới `n − i`, thì ngay truy vấn đầu ô `n` **đang chứa giá trị 1** ⇒ hai phần tử chung một ô, thứ tự bị phá. Ví dụ 1 tình cờ vẫn đúng, nhưng `[2, 2]`: sau truy vấn đầu, 2 và 1 cùng nằm ô `n`, nên truy vấn thứ hai đếm cả 1 là "đứng trước" ⇒ ra 1 thay vì 0.

In [23]:
def wrong_too_few_slots(queries, m):
    n = len(queries)
    fw = Fenwick(n - 1 + m)                    # SAI: chỉ chừa n - 1 ô trống
    where = [0] * (m + 1)
    for v in range(1, m + 1):
        where[v] = n - 1 + v; fw.update(where[v], 1)
    res = []
    for i, q in enumerate(queries):
        p = where[q]
        res.append(fw.query(p) - 1)
        fw.update(p, -1)
        where[q] = n - i                       # i = 0 -> ô n trùng ô của giá trị 1
        fw.update(n - i, 1)
    return res

for q, m in [([3, 1, 2, 1], 5), ([2, 2], 3), ([4, 4], 4)]:
    print(q, m, "| sai:", wrong_too_few_slots(q, m), "| đúng:", process_list(q, m))

[3, 1, 2, 1] 5 | sai: [2, 1, 2, 1] | đúng: [2, 1, 2, 1]
[2, 2] 3 | sai: [1, 1] | đúng: [1, 0]
[4, 4] 4 | sai: [3, 1] | đúng: [3, 0]


**Sửa:** cấp đủ `n` ô trống phía trước (giá trị `v` ở ô `n + v`), ô mới của truy vấn `i` là `n − i ∈ [1, n]`. Lưu ý thêm: không bao giờ dùng ô 0 — `update(0, …)` lặp vô hạn vì `0 & −0 = 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lấy phần tử ra, đưa lên đầu/cuối, hỏi vị trí"** ⇒ move-to-front; ràng buộc nhỏ thì mô phỏng list.
- Hỏi "**vị trí / thứ hạng hiện tại**" khi phần tử liên tục bị dời ⇒ Fenwick đếm số phần tử đứng trước.
- "Đưa lên đầu" nhiều lần ⇒ **chừa sẵn ô trống phía trước** (vị trí ảo), ghi vào ô `n − i`.
- Không cần giữ dãy, chỉ cần vị trí ⇒ giữ **mảng nghịch đảo** `pos`.
- Move-to-front ⇔ ngăn xếp LRU ⇒ vị trí = số phần tử phân biệt truy cập kể từ lần cuối.

## 15. Biến thể của bài

1. **Đưa xuống cuối** thay vì lên đầu ⇒ chừa ô trống phía **sau**, ô mới `m + i + 1`.
2. **Trả về hoán vị cuối cùng** ⇒ các giá trị đã truy vấn theo thứ tự lần truy vấn cuối giảm dần, rồi các giá trị chưa chạm theo thứ tự tăng (Bài 4 mục 17).
3. **`n`, `m` tới 10⁵** ⇒ bắt buộc Fenwick/Segment tree (Bài 5).
4. **LRU Cache** (LeetCode 146): cùng tinh thần move-to-front nhưng cần `O(1)` ⇒ `OrderedDict`/danh sách liên kết đôi + hash map.
5. **Đếm "khoảng cách ngăn xếp"** cho chuỗi truy cập bộ nhớ (phân tích cache) — chính là Cách 4 tăng tốc bằng Fenwick theo thời gian.
6. Bài họ hàng Fenwick: LeetCode 315 *Count of Smaller Numbers After Self*, 2179 *Count Good Triplets in an Array*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng list | `index` + `pop` + `insert(0)` | `O(n·m)` | `O(m)` | ràng buộc nhỏ (như đề) — **ngắn nhất** |
| 2. Mảng vị trí | dịch `pos[v] < p` | `O(n·m)` | `O(m)` | bước đệm, khi không muốn thao tác dãy |
| 3. Fenwick vị trí ảo | đặt vào ô `n − i`, đếm phía trước | `O((n+m) log)` | `O(n+m)` | `n, m` lớn — **tổng quát nhất** |
| 4. Giá trị phân biệt | LRU: đếm giá trị khác nhau từ lần cuối | `O(n²)` | `O(n)` | hiểu bản chất, liên hệ cache |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`processQueries([1, 1, 1], 3)` và `processQueries([2, 3, 2], 3)` trả về gì?

### Bài 2 (tính tay)
Với `queries = [4, 4, 1, 3]`, `m = 4`: viết `P` sau mỗi truy vấn và mảng kết quả.

### Bài 3 (điền TODO)
Hoàn thiện `process_pos_todo` (Cách 2).

### Bài 4 (tự viết lại)
Viết `final_permutation(queries, m)` trả về `P` sau khi xử lý hết, trong `O(n + m)` (không mô phỏng).

### Bài 5 (mini challenge)
Biến thể "**đưa xuống cuối**": mỗi truy vấn trả vị trí hiện tại của `q` rồi chuyển `q` xuống **cuối** `P`. Viết `process_to_back(queries, m)` bằng Fenwick vị trí ảo và kiểm chứng bằng mô phỏng.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def process_pos_todo(queries, m):
    pos = list(range(-1, m))
    res = []
    for q in queries:
        # TODO: p = pos[q]; ghi p
        # TODO: dịch lùi mọi v có pos[v] < p
        # TODO: pos[q] = 0
        pass
    return res

# Bài 4 — TODO
def final_permutation(queries, m):
    # TODO: duyệt queries từ cuối về đầu, lấy các giá trị chưa gặp
    # TODO: nối thêm các giá trị chưa từng truy vấn theo thứ tự tăng
    pass

# Bài 5 — TODO
def process_to_back(queries, m):
    # TODO: ô 1..m cho hoán vị ban đầu, ô m+1..m+n trống; truy vấn i đặt vào ô m + i + 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1,1,1]`: 1 luôn ở đầu ⇒ `[0,0,0]`. (2) `[2,3,2]`: `P=[1,2,3]` → 2 ở 1 → `[2,1,3]` → 3 ở 2 → `[3,2,1]`. (3) 2 ở 1 ⇒ `[1,2,1]`.

**Bài 2.** (1) 4 ở 3 → `[4,1,2,3]`. (2) 4 ở 0 → không đổi; 1 ở 1 → `[1,4,2,3]`. (3) 3 ở 3 → `[3,1,4,2]` ⇒ kết quả `[3,0,1,3]`.

**Bài 3.** (1) `p = pos[q]; res.append(p)`. (2) `for v in range(1, m+1): if pos[v] < p: pos[v] += 1`. (3) `pos[q] = 0`.

**Bài 4.** (1) Giá trị truy vấn sau cùng ở đầu. (2) Duyệt ngược, dùng `set` đánh dấu, mỗi giá trị lấy một lần. (3) Thêm `v` chưa truy vấn theo thứ tự tăng; đối chiếu với mô phỏng.

**Bài 5.** (1) `where[v] = v`, đánh dấu ô `1..m`. (2) Kết quả `query(p) − 1`; xoá ô cũ. (3) Ô mới `m + i + 1` luôn đứng sau mọi phần tử đang có.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
assert Solution().processQueries([1, 1, 1], 3) == [0, 0, 0]
assert Solution().processQueries([2, 3, 2], 3) == [1, 2, 1]
assert Solution().processQueries([4, 4, 1, 3], 4) == [3, 0, 1, 3] == process_list([4, 4, 1, 3], 4)
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [26]:
# Bài 3 — đáp án
def process_pos_sol(queries, m):
    pos = list(range(-1, m))
    res = []
    for q in queries:
        p = pos[q]
        res.append(p)
        for v in range(1, m + 1):
            if pos[v] < p:
                pos[v] += 1
        pos[q] = 0
    return res

rng = random.Random(3)
for _ in range(300):
    m = rng.randint(1, 20); q = [rng.randint(1, m) for _ in range(rng.randint(1, m))]
    assert process_pos_sol(q, m) == process_list(q, m)
print("Bài 3: đúng")

Bài 3: đúng


In [27]:
# Bài 4 — đáp án
def final_permutation_sol(queries, m):
    seen = set()
    front = []
    for q in reversed(queries):
        if q not in seen:
            seen.add(q)
            front.append(q)
    return front + [v for v in range(1, m + 1) if v not in seen]

def final_by_simulation(queries, m):
    P = list(range(1, m + 1))
    for q in queries:
        P.insert(0, P.pop(P.index(q)))
    return P

print(final_permutation_sol([7, 5, 5, 8, 3], 8))
rng = random.Random(4)
for _ in range(500):
    m = rng.randint(1, 25); q = [rng.randint(1, m) for _ in range(rng.randint(1, m))]
    assert final_permutation_sol(q, m) == final_by_simulation(q, m)
print("Bài 4: đúng")

[3, 8, 5, 7, 1, 2, 4, 6]
Bài 4: đúng


In [28]:
# Bài 5 — đáp án
def process_to_back_sol(queries, m):
    n = len(queries)
    fw = Fenwick(n + m)
    where = list(range(m + 1))
    for v in range(1, m + 1):
        fw.update(v, 1)
    res = []
    for i, q in enumerate(queries):
        p = where[q]
        res.append(fw.query(p) - 1)
        fw.update(p, -1)
        where[q] = m + i + 1
        fw.update(m + i + 1, 1)
    return res

def to_back_simulation(queries, m):
    P = list(range(1, m + 1))
    res = []
    for q in queries:
        i = P.index(q)
        res.append(i)
        P.append(P.pop(i))
    return res

rng = random.Random(5)
for _ in range(500):
    m = rng.randint(1, 25); q = [rng.randint(1, m) for _ in range(rng.randint(1, m))]
    assert process_to_back_sol(q, m) == to_back_simulation(q, m)
print(process_to_back_sol([3, 1, 2, 1], 5))
print("Bài 5: đúng")

[2, 0, 0, 3]
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template mô phỏng (ràng buộc nhỏ):**

```python
P = list(range(1, m + 1))
for q in queries:
    i = P.index(q); res.append(i)
    P.insert(0, P.pop(i))
```

**Template Fenwick vị trí ảo (đưa lên đầu):**

```python
size = n + m
where[v] = n + v; update(n + v, 1)          # n ô trống phía trước
for i, q in enumerate(queries):
    p = where[q]; res.append(query(p) - 1)
    update(p, -1); where[q] = n - i; update(n - i, 1)
```

**Checklist trước khi submit:**
- [ ] `P` khởi tạo `1..m` (không phải `0..m−1`).
- [ ] Ghi vị trí **trước** khi di chuyển, đánh số từ 0.
- [ ] Mô phỏng: `pop` rồi `insert(0, …)` — không để bản sao.
- [ ] Mảng vị trí: dịch `pos[v] < p` trước, rồi `pos[q] = 0`.
- [ ] Fenwick: đủ `n + m` ô, ô mới `n − i` (≥ 1), chỉ số từ 1.
- [ ] Đúng chữ ký: `processQueries(self, queries: List[int], m: int) -> List[int]`.